In [1]:
# ── Cell 1: setup ─────────────────────────────────────────────────────────
REPO   = "https://github.com/abirmazumdar03/oct-screening.git"
BRANCH = "main"          # the new Lite0 / MobileNetV2 configs live on main

!git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/working/oct-screening
%cd /kaggle/working/oct-screening

!pip install -q uv
!uv pip install --system -q -e . --no-deps
!uv pip install --system -q -r requirements/kaggle_extra.txt
!ls configs/

/kaggle/working/oct-screening
data.yaml      student_kd_lite0.yaml  student_lite0.yaml
i18n	       student_kd_mnv2.yaml   student.yaml
optimize.yaml  student_kd.yaml	      teacher.yaml


In [2]:
# ── Cell 2: dataset + teacher + output folder (all found automatically) ───
import os, glob, torch

# 1) dataset root = the folder that contains train/CNV (e.g. ".../OCT2017 " with a trailing space)
roots = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train", recursive=True)
               if os.path.isdir(os.path.join(p, "CNV")))
print("dataset roots found:", roots)
assert roots, "Dataset not attached: Add Input → search 'kermany2018'"
os.environ["OCT_DATA_ROOT"] = roots[0]

# 2) teacher checkpoint (Aditya's Kaggle model 'teacher-output', or your own upload with 'teacher'/'r50' in its name)
teachers = [p for p in glob.glob("/kaggle/input/**/best.pt", recursive=True)
            if "teacher" in p.lower() or "r50" in p.lower()]
print("teacher candidates:", teachers)
assert len(teachers) >= 1, "Teacher not attached: Add Input → Models → teacher-output (or oct-teacher-r50)"
os.environ["OCT_TEACHER_CKPT"] = teachers[0]

# 3) where results are written (shows up in the notebook's Output tab)
os.environ["OCT_OUT_DIR"] = "/kaggle/working/artifacts/runs"

print("OCT_DATA_ROOT   :", repr(os.environ["OCT_DATA_ROOT"]))
print("OCT_TEACHER_CKPT:", os.environ["OCT_TEACHER_CKPT"])
print("GPU             :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE: Settings → Accelerator → GPU")

dataset roots found: ['/kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 ', '/kaggle/input/datasets/paultimothymooney/kermany2018/oct2017/OCT2017 ', '/kaggle/input/datasets/paultimothymooney/kermany2018/oct2017/__MACOSX/OCT2017 ']
teacher candidates: ['/kaggle/input/models/abirmazumdar/oct-teacher-r50-v1/pytorch/default/1/best.pt']
OCT_DATA_ROOT   : '/kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 '
OCT_TEACHER_CKPT: /kaggle/input/models/abirmazumdar/oct-teacher-r50-v1/pytorch/default/1/best.pt
GPU             : Tesla T4


In [3]:
# ── Cell 3: train the student WITH the teacher (KD), then evaluate on val ──
# Expect in the log:  arch=efficientnet_lite0.ra_in1k  device=cuda
#                     [kd] teacher=resnet50.tv2_in1k (val macro-F1 0.9238)
!python -m octscreen.training.train --config configs/student_kd_lite0.yaml
!python -m octscreen.training.evaluate --checkpoint /kaggle/working/artifacts/runs/student_kd_lite0_v1/best.pt --split val

[run] student_kd_lite0_v1 | arch=efficientnet_lite0.ra_in1k | device=cuda amp=True | out=/kaggle/working/artifacts/runs/student_kd_lite0_v1
[data] train=67807 val=7980 batches/epoch=529

model.safetensors: downloading bytes: ████████████████████▉ | 17.9MB,  760kB/s  
model.safetensors: downloading bytes: ██████████████████████| 17.9MB, 1.74MB/s  
model.safetensors: reconstructing file: 100%|██████| 18.8MB / 18.8MB, 1.82MB/s  
[kd] teacher=resnet50.tv2_in1k (val macro-F1 0.9238) T=4.0 alpha=0.7
  ep0 step 0/529 loss=13.0876
  ep0 step 100/529 loss=2.3667
  ep0 step 200/529 loss=1.4719
  ep0 step 300/529 loss=1.5521
  ep0 step 500/529 loss=0.7942
[epoch] epoch=0 train_loss=2.021 train_ce=0.6329 train_kl=2.616 val_loss=0.4577 val_acc=0.9093 val_macro_f1=0.8625 val_urgent_sens=0.958 lr_backbone=0.0009905 lr_head=0.001981 epoch_sec=1591
  ↳ new best macro_f1=0.8625 saved
  ep1 step 0/529 loss=0.6175
  ep1 step 100/529 loss=0.6521
  ep1 step 200/529 loss=0.3783
  ep1 step 300/529 loss=0.4469